<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Autonomous Traders</h2>
            <span style="color:#ff7800;">An equity trading simulation to illustrate autonomous agents powered by tools and resources from MCP servers.
            </span>
        </td>
    </tr>
</table>

### Week 6 Day 4

And now - introducing the Capstone project:


# Autonomous Traders

An equity trading simulation, with 4 Traders and a Researcher, powered by a slew of MCP servers with tools & resources:

1. Our home-made Accounts MCP server (written by our engineering team!)
2. Fetch (get webpage via a local headless browser)
3. Memory
4. Brave Search
5. Financial data

And a resource to read information about the trader's account, and their investment strategy.

The goal of today's lab is to make a new python module, `traders.py` that will manage a single trader on our trading floor.

We will experiment and explore in the lab, and then migrate to a python module when we're ready.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">One more time --</h2>
            <span style="color:#ff7800;">Please do not use this for actual trading decisions!!
            </span>
        </td>
    </tr>
</table>

In [1]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

True

### Let's start by gathering the MCP params for our trader

In [5]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

False
False


In [6]:
if is_paid_polygon or is_realtime_polygon:
    market_mcp = {"command": "uvx","args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"], "env": {"POLYGON_API_KEY": polygon_api_key}}
else:
    market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### And now for our researcher

In [7]:
serper_env = {"SERPER_API_KEY": os.getenv("SERPER_API_KEY")}

researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "uv", "args": ["run", "serper_server.py"], "env": serper_env}
]

### Now create the MCPServerStdio for each

In [8]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### Now let's make a Researcher Agent to do market research

And turn it into a tool - remember how this works for OpenAI Agents SDK, and the difference with handoffs?

In [9]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""You are a financial researcher. You are able to search the web for interesting financial news,
look for possible trading opportunities, and help with research.
Based on the request, you carry out necessary research and respond with your findings.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with investment opportunities based on searching latest news.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
    )
    return researcher

In [10]:
research_question = "What's the latest news on Amazon?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



Here's the latest news on Amazon:

1. Amazon has added $1.4 billion to its Housing Equity Fund to create an additional 14,000 affordable homes. The company also met its 100% renewable energy goal seven years early.
2. Amazon is cutting 14,000 corporate jobs. CEO Andy Jassy explained that the layoffs are not due to financial reasons or AI, but rather related to company culture and the need for agility.
3. Amazon is starting to offer prescription drugs through vending machines, with new self-service kiosks allowing One Medical patients in Los Angeles to pick up medications.
4. The CEO emphasized that the massive layoffs were about making the company more agile and were not driven by cost-cutting or AI automation.
5. Amazon recently reported Q3 earnings that beat expectations on both top and bottom lines, driven by growth in its cloud business (AWS).

If you want, I can provide more details on any of these points or look into specific aspects further.

In [11]:
research_question = "What's the latest news on Amazon?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



Here are the latest news highlights on Amazon:

1. Amazon added $1.4 billion to its Housing Equity Fund to create an additional 14,000 affordable homes. They have also met their 100% renewable energy goal seven years early.

2. Amazon is cutting 14,000 corporate jobs, but CEO Andy Jassy states the reason is related to "culture" and agility rather than money or AI-related reasons.

3. Amazon is starting to offer prescription drugs through vending machines as part of a new self-service kiosk pilot for Amazon One Medical patients in Los Angeles.

4. Amazon's CEO reiterated that the recent layoffs are about making the company more agile and are not driven by financial strain or cost-cutting.

5. Amazon reported strong Q3 earnings, beating expectations on both revenue and profit, with growth in its AWS cloud business driving stock gains.

If you want, I can provide more details on any of these news points or look into specific financial metrics or trading implications from these updates.

### Look at the trace

https://platform.openai.com/traces

In [13]:
muadh_initial_strategy = "You are a day trader that aggressively buys and sells shares based on news and market conditions."
Account.get("muadh").reset(muadh_initial_strategy)

display(Markdown(await read_accounts_resource("muadh")))
display(Markdown(await read_strategy_resource("muadh")))

{"name": "muadh", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-10-31 06:49:59", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

You are a day trader that aggressively buys and sells shares based on news and market conditions.

### And now - to create our Trader Agent

In [14]:
agent_name = "muadh"

# Using MCP Servers to read resources
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
{strategy}
Your current holdings and balance is:
{account_details}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Please make use of these tools to manage your portfolio. Carry out trades as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [15]:
print(instructions)


You are a trader that manages a portfolio of shares. Your name is muadh and your account is under your name, muadh.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
You are a day trader that aggressively buys and sells shares based on news and market conditions.
Your current holdings and balance is:
{"name": "muadh", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-10-31 06:49:59", 10000.0], ["2025-10-31 06:50:18", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so fa

### And to run our Trader

In [20]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="This tool researches online for news and opportunities, \
                either based on your specific request to look into a certain stock, \
                or generally for notable financial news and opportunities. \
                Describe what kind of research you're looking for."
        )

In [21]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="gpt-4o-mini",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

### Summary of Actions:

1. **Market Research**: 
   - Investigated the latest stock market news and trends which highlighted a rebound in tech stocks due to strong earnings reports from Apple and Amazon.
   - Noted an overall optimistic market sentiment which could present good trading opportunities.

2. **Stock Price Lookup**:
   - Current prices before transactions:
     - **Apple (AAPL)**: $271.40
     - **Amazon (AMZN)**: $222.86

3. **Initial Trades**:
   - **AAPL**: Bought 30 shares at an average price of $271.94.
   - **Attempted to buy 40 shares of AMZN** but failed due to insufficient funds after the AAPL purchase.

4. **Adjustment**:
   - Sold 10 shares of AAPL at an average price of $270.86 to take partial profits, recalibrating the portfolio.

### Current Portfolio Details:
- **Remaining Holdings**: 
  - 20 shares of AAPL
- **Balance**: $4,550.29
- **Total Portfolio Value**: $9,978.29
- **Total Profit/Loss**: -$21.71

### Next Steps:
- Continue monitoring the tech sector for further trading opportunities.
- Consider re-investing in AMZN or other opportunities as the balance allows.

### Then go and look at the trace

http://platform.openai.com/traces


In [22]:
# And let's look at the results of the trading

await read_accounts_resource(agent_name)

'{"name": "muadh", "balance": 4550.2880000000005, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"AAPL": 20}, "transactions": [{"symbol": "AAPL", "quantity": 30, "price": 271.9428, "timestamp": "2025-10-31 06:54:39", "rationale": "Strong earnings report and positive market sentiment, ideal for short-term trading."}, {"symbol": "AAPL", "quantity": -10, "price": 270.8572, "timestamp": "2025-10-31 06:54:42", "rationale": "Taking partial profits after short-term price movements based on recent earnings."}], "portfolio_value_time_series": [["2025-10-31 06:49:59", 10000.0], ["2025-10-31 06:50:18", 10000.0], ["2025-10-31 06:54:39", 9983.716], ["2025-10-31 06:54:42", 9978.288], ["2025-10-31 06:55:21", 9978.288]], "total_portfolio_value": 9978.288, "total_profit_loss": -21.711999999999534}'

### Now it's time to review the Python module made from this:

`mcp_params.py` is where the MCP servers are specified. You'll notice I've brought in some familiar friends: memory and push notifications!

`templates.py` is where the instructions and messages are set up (i.e. the System prompts and User prompts)

`traders.py` brings it all together.

You'll notice I've done something a bit fancy with code like this:

```
async with AsyncExitStack() as stack:
    mcp_servers = [await stack.enter_async_context(MCPServerStdio(params)) for params in mcp_server_params]
```

This is just a tidy way to combine our "with" statements (known as context managers) so that we don't need to do something ugly like this:

```
async with MCPServerStdio(params=params1) as mcp_server1:
    async with MCPServerStdio(params=params2) as mcp_server2:
        async with MCPServerStdio(params=params3) as mcp_server3:
            mcp_servers = [mcp_server1, mcp_server2, mcp_server3]
```

But it's equivalent.


In [6]:
from traders import Trader


In [7]:
trader = Trader("muadh")

In [3]:
import os
# point to your nvm-installed node and preload polyfill
os.environ["PATH"] = "/home/muadh/.nvm/versions/node/v20.19.5/bin:" + os.environ.get("PATH", "")
os.environ["NODE_OPTIONS"] = "--require /home/muadh/agents/6_mcp/node_polyfills.js"

# verify within the notebook kernel
!which node
!node -v
!echo $NODE_OPTIONS

/home/muadh/.nvm/versions/node/v20.19.5/bin/node
v20.19.5
--require /home/muadh/agents/6_mcp/node_polyfills.js


In [4]:
!node -v   


v20.19.5


In [8]:
await trader.run()

In [9]:
await read_accounts_resource("muadh")

'{"name": "muadh", "balance": 94.7447000000002, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"AAPL": 20, "META": 5, "AMZN": 5}, "transactions": [{"symbol": "AAPL", "quantity": 30, "price": 271.9428, "timestamp": "2025-10-31 06:54:39", "rationale": "Strong earnings report and positive market sentiment, ideal for short-term trading."}, {"symbol": "AAPL", "quantity": -10, "price": 270.8572, "timestamp": "2025-10-31 06:54:42", "rationale": "Taking partial profits after short-term price movements based on recent earnings."}, {"symbol": "META", "quantity": 5, "price": 667.80294, "timestamp": "2025-10-31 09:20:01", "rationale": "Meta\'s strong potential for recovery post-earnings drop; targeting short-term gains."}, {"symbol": "AMZN", "quantity": 5, "price": 223.30572, "timestamp": "2025-10-31 09:20:01", "rationale": "Anticipating positive earnings, Amazon could experience upward momentum; ideal for short-term tr

### Now look at the trace

https://platform.openai.com/traces

### How many tools did we use in total?

In [10]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params

all_params = trader_mcp_server_params + researcher_mcp_server_params("ed")

count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"We have {len(all_params)} MCP servers, and {count} tools")

We have 6 MCP servers, and 15 tools
